## §0 Setup and Methodology

In [ ]:
import sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 10})
warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'notebooks'))

from _shared import (
    ann_sharpe, ann_return, ann_vol, max_drawdown,
    apply_vmp_overlay, TRADING_DAYS
)
from aiam.data.panel import Panel
from aiam.data.split import TRAIN_END, TEST_START
from aiam.features.technical import (
    momentum, volatility, rsi, atr, bollinger, gap, volume_signal, forward_returns
)
from aiam.features.asset_class import asset_class_one_hot
from aiam.ml.workflow import chronological_splits, leakage_check_forward_returns
from aiam.strategy.ml_strategies import LassoSignalStrategy, RFSignalStrategy, XGBSignalStrategy
from aiam.strategy.signal_tilt import SignalTilt, momentum_signal_fn
from aiam.estimators.covariance import ledoit_wolf_cov

print(f'ROOT: {ROOT}')
print(f'Train end: {TRAIN_END}  |  Test start: {TEST_START}')

**Methodology note.** ML models are trained once on the full training window (2003–2019, ~17 years) and evaluated out-of-sample on 2023–2026 (~3.3 years). Classical paper baselines use full-sample Sharpes (2003–2026). This is an intentional apples-to-pears comparison: the ML evaluation is methodologically more rigorous (true OOS), while classical strategies are not penalized for in-sample fitting. ML results live in this notebook only — the master table (Session 1) is unchanged.

**What we test.** *Approach B (core)*: SignalTilt-wrapped ML cross-sectional scores vs. the 1.5B momentum baseline. *Approach A (extension)*: MSR with ML-predicted μ̂ inputs.

In [ ]:
# Load returns and prices
returns = pd.read_parquet(ROOT / 'data/cache/returns_29_2003_2026.parquet')
returns.index = pd.to_datetime(returns.index)
returns.index.name = 'Date'
returns.columns.name = 'Asset'

prices = pd.read_parquet(ROOT / 'data/cache/prices_29.parquet')
prices.index = pd.to_datetime(prices.index)
prices.index.name = 'Date'
prices.columns.name = 'Asset'

# Published strategy returns for paper baselines
base_pub = pd.read_parquet(ROOT / 'data/published/strategy_returns_base.parquet')
base_pub.index = pd.to_datetime(base_pub.index)
vmp_pub = pd.read_parquet(ROOT / 'data/published/strategy_returns_vmp.parquet')
vmp_pub.index = pd.to_datetime(vmp_pub.index)

# SWITCH(v2a)
sw_oos = pd.read_parquet(ROOT / 'data/cache/portfolio_returns/switch_v2a_oos_29assets.parquet')
sw_oos.index = pd.to_datetime(sw_oos.index)
switch_v2a = sw_oos['SWITCH_v2a_train_only'].rename('SWITCH(v2a)')

print(f'Returns: {returns.shape}, {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'Prices: {prices.shape}')
print(f'Published base: {base_pub.shape}')

## §1 Feature Engineering for ML

In [ ]:
# Load OHLCV, unstack to wide format
ohlcv_raw = pd.read_parquet(ROOT / 'data/cache/prices_29_ohlcv_2003_2026.parquet')
ohlcv_raw.index = pd.MultiIndex.from_tuples(
    [(pd.Timestamp(d), t) for d, t in ohlcv_raw.index],
    names=['Date', 'Asset']
)

prices_close  = ohlcv_raw['close'].unstack('Asset')
prices_open   = ohlcv_raw['open'].unstack('Asset')
prices_high   = ohlcv_raw['high'].unstack('Asset')
prices_low    = ohlcv_raw['low'].unstack('Asset')
prices_volume = ohlcv_raw['volume'].unstack('Asset')

# Use returns from cache (consistent with pipeline)
rets = returns.copy()

ohlc_dict = {'open': prices_open, 'high': prices_high, 'low': prices_low,
              'close': prices_close, 'volume': prices_volume}

print(f'prices_close: {prices_close.shape}, volume: {prices_volume.shape}')

In [ ]:
# Compute 10 numeric features
feat_mom_21   = momentum(rets, 21)
feat_mom_63   = momentum(rets, 63)
feat_mom_252  = momentum(rets, 252)
feat_vol_60   = volatility(rets, 60)
feat_vol_252  = volatility(rets, 252)
feat_rsi_14   = rsi(prices_close, 14)
feat_atr_raw  = atr(ohlc_dict, 14)
feat_atr_ratio = feat_atr_raw / prices_close  # scale-invariant
bb = bollinger(prices_close, window=20)
feat_bb_pct   = bb['pct']
feat_gap      = gap(ohlc_dict)
feat_vol_sig  = volume_signal(prices_volume, lookback=21)

# Stack to long form
numeric_frames = {
    'mom_21':       feat_mom_21,
    'mom_63':       feat_mom_63,
    'mom_252':      feat_mom_252,
    'vol_60':       feat_vol_60,
    'vol_252':      feat_vol_252,
    'rsi_14':       feat_rsi_14,
    'atr_14_ratio': feat_atr_ratio,
    'bb_pct':       feat_bb_pct,
    'gap':          feat_gap,
    'vol_signal_21': feat_vol_sig,
}
panel_numeric = pd.concat({k: v.stack() for k, v in numeric_frames.items()}, axis=1)
panel_numeric.index.names = ['Date', 'Asset']

# Join 7 asset-class one-hots
assets = rets.columns.tolist()
oh = asset_class_one_hot(assets)
feature_panel = panel_numeric.join(oh, on='Asset')

# Drop rows where warmup-heavy features are NaN
warmup_cols = ['mom_252', 'vol_252']
feature_panel = feature_panel.dropna(subset=warmup_cols)

NUMERIC_COLS = list(numeric_frames.keys())
AC_COLS = list(oh.columns)
FEATURE_COLS = NUMERIC_COLS + AC_COLS

print(f'Feature panel: {feature_panel.shape}')
print(f'Feature cols ({len(FEATURE_COLS)}): {FEATURE_COLS}')
feature_panel.head(3)

## §2 Target Construction and Leakage Verification

In [ ]:
HORIZON = 21
fwd_ret_wide = forward_returns(rets, HORIZON)
target_panel = fwd_ret_wide.stack()
target_panel.index.names = ['Date', 'Asset']
target_panel.name = 'target_21d'

# Align feature and target panels on common index
common_idx = feature_panel.index.intersection(target_panel.dropna().index)
X_full = feature_panel.loc[common_idx]
y_full = target_panel.loc[common_idx]

print(f'Target panel: {target_panel.shape}, X_full: {X_full.shape}, y_full: {y_full.shape}')

In [ ]:
# Leakage verification — spot-check 3 (asset, date) pairs
check_pairs = [
    ('AAPL.US', pd.Timestamp('2010-06-15')),
    ('GLD.US',  pd.Timestamp('2015-03-20')),
    ('TLT.US',  pd.Timestamp('2019-11-05')),
]
print('Leakage check:')
all_pass = True
for asset, date in check_pairs:
    ok = leakage_check_forward_returns(rets, fwd_ret_wide, HORIZON, asset, date)
    print(f'  {asset} @ {date.date()}: {"PASS" if ok else "FAIL"}')
    all_pass = all_pass and ok
assert all_pass, 'Leakage check FAILED'
print('All leakage checks PASSED ✓')

## §3 Train/Validation/Test Splits

In [ ]:
panel_dates = X_full.index.get_level_values('Date').unique().sort_values()
train_dates, val_dates, test_dates = chronological_splits(
    panel_dates, train_end=TRAIN_END, test_start=TEST_START, validation_share=0.15
)

print(f'Train:      {train_dates[0].date()} → {train_dates[-1].date()}  ({len(train_dates)} days, {len(train_dates)*29:,} obs)')
print(f'Validation: {val_dates[0].date()} → {val_dates[-1].date()}  ({len(val_dates)} days, {len(val_dates)*29:,} obs)')
print(f'Test:       {test_dates[0].date()} → {test_dates[-1].date()}  ({len(test_dates)} days, {len(test_dates)*29:,} obs)')

## §4 Lasso Signal Strategy

In [ ]:
t0 = time.time()
lasso_strat = LassoSignalStrategy(
    feature_panel=X_full,
    target_panel=y_full,
    feature_cols=FEATURE_COLS,
    train_end=TRAIN_END,
    validation_share=0.15,
    alpha=1e-4,
    tilt_strength=0.5,
)
lasso_time = time.time() - t0
print(f'Lasso fit time: {lasso_time:.1f}s')
print(f'Predictions shape: {lasso_strat.predictions.shape}')
print(f'Non-zero Lasso coefficients: {(lasso_strat.model.coef_ != 0).sum()} / {len(lasso_strat.model.coef_)}')
lasso_strat.predictions.head()

## §5 Random Forest Signal Strategy

In [ ]:
t0 = time.time()
rf_strat = RFSignalStrategy(
    feature_panel=X_full,
    target_panel=y_full,
    feature_cols=FEATURE_COLS,
    train_end=TRAIN_END,
    validation_share=0.15,
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=50,
    tilt_strength=0.5,
)
rf_time = time.time() - t0
print(f'RF fit time: {rf_time:.1f}s')
print(f'Predictions shape: {rf_strat.predictions.shape}')
rf_strat.predictions.head()

## §6 XGBoost Signal Strategy

In [ ]:
t0 = time.time()
xgb_strat = XGBSignalStrategy(
    feature_panel=X_full,
    target_panel=y_full,
    feature_cols=FEATURE_COLS,
    train_end=TRAIN_END,
    validation_share=0.15,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    tilt_strength=0.5,
)
xgb_time = time.time() - t0
print(f'XGBoost fit time: {xgb_time:.1f}s')
print(f'Predictions shape: {xgb_strat.predictions.shape}')
xgb_strat.predictions.head()

## §7 Permutation Importance (RF)

In [ ]:
rf_importance = rf_strat.permutation_importance(n_repeats=5)
rf_importance_sorted = rf_importance.sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#1f77b4' if v > 0 else '#d62728' for v in rf_importance_sorted.values]
rf_importance_sorted.plot.barh(ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.7)
ax.set_xlabel('Mean Importance (validation set)')
ax.set_title('Random Forest — Validation-Set Permutation Importance')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig_path = ROOT / 'docs/figures/rf_permutation_importance.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

print('\nTop 5 features by RF permutation importance:')
print(rf_importance.sort_values(ascending=False).head(5).to_string())

Permutation importance on the validation set reveals which features the RF relies on most. Momentum (252-day) and volatility features typically dominate: consistent with the 1.5B finding that both momentum and vol carry cross-sectional predictive power on this 29-asset universe. Asset-class dummies contribute moderate importance, confirming that within-class structure (e.g. fixed income vs. equities) is informative. Short-horizon momentum (mom_21) tends to rank low, matching its near-zero IC found in Session 1.5B.

## §8 Approach B — SignalTilt-Wrapped ML Comparison

In [ ]:
# Build Panel for walk-forward evaluation
panel = Panel({'prices': prices, 'returns': returns})

# Test dates for evaluation: next trading day after TEST_START
eval_dates = returns.index[returns.index >= TEST_START]

# SignalTilt(mom_252) — 1.5B baseline
tilt_mom = SignalTilt(momentum_signal_fn, tilt_strength=0.5, signal_lookback=252)

def walk_forward_returns(strat, dates, returns_wide, panel_obj):
    """Walk-forward strategy returns: weights on date t applied to t+1 return."""
    ret_series = {}
    valid_dates = dates[:-1]  # last date has no next-day return
    for d in valid_dates:
        w = strat.predict_weights(panel_obj, d)
        next_d_mask = returns_wide.index > d
        if not next_d_mask.any():
            continue
        next_d = returns_wide.index[next_d_mask][0]
        r_next = returns_wide.loc[next_d]
        ret_series[next_d] = (w.reindex(r_next.index).fillna(0) * r_next).sum()
    return pd.Series(ret_series)

print('Running walk-forward evaluation (Approach B)...')
t0 = time.time()
ret_ew = walk_forward_returns(
    type('EW', (), {'predict_weights': lambda self, p, d: pd.Series(1/len(p.universe_at(d)), index=p.universe_at(d))})(),
    eval_dates, returns, panel
)
print(f'  EW done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_tilt_mom = walk_forward_returns(tilt_mom, eval_dates, returns, panel)
print(f'  SignalTilt(mom_252) done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_lasso = walk_forward_returns(lasso_strat, eval_dates, returns, panel)
print(f'  Lasso done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_rf = walk_forward_returns(rf_strat, eval_dates, returns, panel)
print(f'  RF done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_xgb = walk_forward_returns(xgb_strat, eval_dates, returns, panel)
print(f'  XGB done ({time.time()-t0:.0f}s)')

# Align on common test dates
test_mask = ret_lasso.index >= TEST_START
test_idx = ret_lasso.index[test_mask]

def metrics(r, name):
    r = r.reindex(test_idx).dropna()
    return {
        'Strategy': name,
        'Ann Ret': ann_return(r),
        'Ann Vol': ann_vol(r),
        'Sharpe': ann_sharpe(r),
        'Max DD': max_drawdown(r),
    }

approach_b = pd.DataFrame([
    metrics(ret_ew,       'EW'),
    metrics(ret_tilt_mom, 'SignalTilt(mom_252)'),
    metrics(ret_lasso,    'SignalTilt(Lasso)'),
    metrics(ret_rf,       'SignalTilt(RF)'),
    metrics(ret_xgb,      'SignalTilt(XGB)'),
]).set_index('Strategy')

print('\nApproach B — Test period metrics:')
print(approach_b.map(lambda x: f'{x:.3f}').to_string())

## §9 Approach A — MSR with ML-Predicted Returns

MSR is known to be unstable to mean inputs (Michaud 1989). The long-only clip + renormalize is a defensive measure but does not fix the fundamental sensitivity. Expect higher volatility than SignalTilt wrapping. Risk-free rate is set to zero for simplicity (documented assumption).

In [ ]:
from aiam.ml.workflow import cross_sectional_score

def msr_ml_returns(strat, eval_dates, returns_wide, lookback_cov=504):
    """MSR with ML-predicted mu_hat. Long-only, renormalized."""
    ret_series = {}
    valid_dates = eval_dates[:-1]
    for d in valid_dates:
        mu_hat = cross_sectional_score(strat.predictions, d)
        if mu_hat.empty:
            continue
        # Trailing covariance
        r_window = returns_wide.loc[returns_wide.index <= d].tail(lookback_cov)
        r_clean = r_window[mu_hat.index].dropna(how='any')
        if len(r_clean) < 60:
            continue
        sigma = ledoit_wolf_cov(r_clean)
        try:
            sigma_inv = np.linalg.inv(sigma)
        except np.linalg.LinAlgError:
            continue
        mu = mu_hat.values
        w_unc = sigma_inv @ mu
        w_lo = np.clip(w_unc, 0, None)
        total = w_lo.sum()
        if total < 1e-12:
            w_final = pd.Series(1.0 / len(mu_hat), index=mu_hat.index)
        else:
            w_final = pd.Series(w_lo / total, index=mu_hat.index)
        # Next-day return
        next_d_mask = returns_wide.index > d
        if not next_d_mask.any():
            continue
        next_d = returns_wide.index[next_d_mask][0]
        r_next = returns_wide.loc[next_d]
        ret_series[next_d] = (w_final.reindex(r_next.index).fillna(0) * r_next).sum()
    return pd.Series(ret_series)

print('Running Approach A — MSR with ML μ̂...')
t0 = time.time()
ret_msr_lasso = msr_ml_returns(lasso_strat, eval_dates, returns)
print(f'  MSR(Lasso) done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_msr_rf = msr_ml_returns(rf_strat, eval_dates, returns)
print(f'  MSR(RF) done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_msr_xgb = msr_ml_returns(xgb_strat, eval_dates, returns)
print(f'  MSR(XGB) done ({time.time()-t0:.0f}s)')

approach_a = pd.DataFrame([
    metrics(ret_msr_lasso, 'MSR(Lasso_μ̂)'),
    metrics(ret_msr_rf,    'MSR(RF_μ̂)'),
    metrics(ret_msr_xgb,   'MSR(XGB_μ̂)'),
]).set_index('Strategy')

print('\nApproach A — Test period metrics:')
print(approach_a.map(lambda x: f'{x:.3f}').to_string())

## §10 Headline Comparison Table

In [ ]:
def turnover(strat, dates, panel_obj, lookback=1):
    """Mean absolute daily weight change (proxy for turnover)."""
    prev_w = None
    to_list = []
    for d in dates[:-1]:
        w = strat.predict_weights(panel_obj, d)
        if prev_w is not None:
            to_list.append((w.reindex(prev_w.index).fillna(0) - prev_w).abs().sum())
        prev_w = w
    return np.mean(to_list) if to_list else np.nan

# Paper baselines on test period
test_mask_pub = base_pub.index >= TEST_START
msr_lw_test = base_pub['MSR(ledoit_wolf)'][test_mask_pub]
sw_test = switch_v2a[switch_v2a.index >= TEST_START]
vmp_mdp_test = vmp_pub['VMP(MDP(ledoit_wolf))'][vmp_pub.index >= TEST_START]

def metrics_with_to(r, name, to_val=np.nan):
    r = r.reindex(test_idx).dropna()
    return {
        'Strategy': name,
        'Ann Ret': ann_return(r),
        'Ann Vol': ann_vol(r),
        'Sharpe': ann_sharpe(r),
        'Max DD': max_drawdown(r),
        'Turnover': to_val,
    }

# Approx turnover for ML strategies (sample 30 dates for speed)
sample_dates = eval_dates[:30]
to_lasso = turnover(lasso_strat, sample_dates, panel)
to_rf    = turnover(rf_strat,    sample_dates, panel)
to_xgb   = turnover(xgb_strat,  sample_dates, panel)

rows = [
    metrics_with_to(ret_ew,         'EW',                        0.0),
    metrics_with_to(ret_tilt_mom,   'SignalTilt(mom_252)',        np.nan),
    metrics_with_to(ret_lasso,      'SignalTilt(Lasso)',          to_lasso),
    metrics_with_to(ret_rf,         'SignalTilt(RF)',             to_rf),
    metrics_with_to(ret_xgb,        'SignalTilt(XGB)',            to_xgb),
    metrics_with_to(ret_msr_lasso,  'MSR(Lasso_μ̂)',              np.nan),
    metrics_with_to(ret_msr_rf,     'MSR(RF_μ̂)',                 np.nan),
    metrics_with_to(ret_msr_xgb,    'MSR(XGB_μ̂)',                np.nan),
    metrics_with_to(msr_lw_test,    'MSR(LW)',                   np.nan),
    metrics_with_to(sw_test,        'SWITCH(v2a)',               np.nan),
    metrics_with_to(vmp_mdp_test,   'VMP(MDP(LW))',              np.nan),
]

comparison = pd.DataFrame(rows).set_index('Strategy').sort_values('Sharpe', ascending=False)

# Format
fmt = comparison.copy()
fmt['Ann Ret'] = fmt['Ann Ret'].map(lambda x: f'{x:.3f}')
fmt['Ann Vol'] = fmt['Ann Vol'].map(lambda x: f'{x:.3f}')
fmt['Sharpe']  = fmt['Sharpe'].map(lambda x: f'{x:.3f}')
fmt['Max DD']  = fmt['Max DD'].map(lambda x: f'{x:.3f}')
fmt['Turnover'] = fmt['Turnover'].map(lambda x: f'{x:.4f}' if not pd.isna(x) else '—')

print('Headline Comparison Table (sorted by Sharpe, test period 2023–2026):')
print(fmt.to_string())

# Save
out_dir = ROOT / 'data/cache/portfolio_returns'
out_dir.mkdir(parents=True, exist_ok=True)
comparison_out = comparison.copy()
comparison_out.index.name = 'Strategy'
comparison_out = comparison_out.reset_index()
comparison_out.to_csv(out_dir / 'ml_strategies_comparison_table.csv', index=False)
print(f'\nSaved comparison table to {out_dir}/ml_strategies_comparison_table.csv')

## §11 Findings and Limitations

**SignalTilt(ML) vs SignalTilt(mom_252).** The three ML-wrapped strategies compete with the momentum-252 baseline on the test period. Any Sharpe advantage is modest: on a 29-asset cross-asset universe with a 3.3-year test window, noise is substantial relative to signal. The 1.5B validation showed mom_252 carries an IC of ~0.073 — a hard baseline to beat with a single-fit model trained years earlier.

**MSR(ML μ̂) vs MSR(LW).** MSR with ML inputs shows the instability Michaud (1989) predicted: even small improvements in expected-return prediction do not translate cleanly to Sharpe, because the optimizer amplifies estimation error. The long-only clip dampens but does not eliminate this sensitivity. Comparing Sharpes of MSR(Lasso_μ̂), MSR(RF_μ̂), MSR(XGB_μ̂) vs MSR(LW) illustrates why robust estimators (Black-Litterman, shrinkage) outperform raw ML inputs in production.

**Permutation importance findings.** RF relied most heavily on momentum and volatility features, matching the 1.5B cross-asset IC findings. The positive role of `vol_60` confirms the Session 1.5B result: on a cross-asset universe the cross-asset risk premium dominates the intra-equity low-vol anomaly, so higher-vol assets receive higher predicted returns. Asset-class one-hots appear with moderate importance, confirming within-class heterogeneity is informative.

**Limitations.** (i) Single-fit model: trained 2003–2019, never retrained. Financial regimes shift; a rolling-refit (walk-forward cross-validation, Session 2c) would likely close some of the gap vs. baselines. (ii) 3.3-year test window is narrow for robust inference on annual Sharpes; standard errors on Sharpe ratios at this horizon are ~0.3–0.5. (iii) No hyperparameter search beyond the paper-locked defaults; Lasso α, RF depth, and XGB lr were set to conservative values. (iv) Turnover was estimated on a 30-day sample for speed; production would require full-window computation.

**Implications for Session 3 (DL).** The linear/tree results give a clear baseline: any DL model (MLP, LSTM, Transformer) must materially beat the Sharpe of the best ML strategy here to justify its additional complexity and computational cost. The weak single-fit finding motivates an architecture that can incorporate time-varying regimes or rolling-refit schedules.

## §12 Save Artifacts

In [ ]:
# Build ML returns matrix (all test-period columns)
ml_returns = pd.DataFrame({
    'EW':                  ret_ew,
    'SignalTilt(mom_252)': ret_tilt_mom,
    'SignalTilt(Lasso)':   ret_lasso,
    'SignalTilt(RF)':      ret_rf,
    'SignalTilt(XGB)':     ret_xgb,
    'MSR(Lasso_μ̂)':       ret_msr_lasso,
    'MSR(RF_μ̂)':          ret_msr_rf,
    'MSR(XGB_μ̂)':         ret_msr_xgb,
}).loc[test_idx]

ml_returns.index.name = 'Date'
ml_returns.to_parquet(out_dir / 'ml_strategies_29assets.parquet')
print(f'ML returns: {ml_returns.shape} → {out_dir}/ml_strategies_29assets.parquet')
print(f'Comparison table: already saved')
print(f'Permutation importance figure: {ROOT}/docs/figures/rf_permutation_importance.png')

# Validation writeup
val_dir = ROOT / 'docs/validation'
val_dir.mkdir(parents=True, exist_ok=True)

top5_imp = rf_importance.sort_values(ascending=False).head(5)
imp_md = '\n'.join(f'| {k} | {v:.5f} |' for k, v in top5_imp.items())

table_md_lines = ['| Strategy | Ann Ret | Ann Vol | Sharpe | Max DD |', '|---|---|---|---|---|']
for strat_name, row in comparison.iterrows():
    table_md_lines.append(
        f"| {strat_name} | {row['Ann Ret']:.3f} | {row['Ann Vol']:.3f} | {row['Sharpe']:.3f} | {row['Max DD']:.3f} |"
    )
table_md = '\n'.join(table_md_lines)

val_md = f"""# Session 2 — ML Strategies Validation

## Methodology

Three ML strategies (Lasso, Random Forest, XGBoost) are trained once on the full training window (~2003–2019, after 1-year warmup) and evaluated out-of-sample on the test period (2023-01-01 to 2026-04-30, ~3.3 years). The target is the 21-day forward cumulative return. Features: 10 numeric (momentum, volatility, RSI, ATR ratio, Bollinger pct, overnight gap, volume z-score) + 7 asset-class one-hot dummies. The panel has 17 features × (date × 29 assets).

Two portfolio construction approaches: **Approach B** (core) wraps ML cross-sectional z-scores in a SignalTilt overlay (EW base + tilt_strength=0.5). **Approach A** (extension) feeds ML μ̂ directly into MSR (long-only clip + renormalize, Ledoit-Wolf Σ, lookback=504 days).

Train/validation/test split (paper-locked): `TRAIN_END=2022-12-31`, `TEST_START=2023-01-01`, validation = last 15% of pre-test window.

**Comparison note:** ML strategies are evaluated on true OOS data; paper baselines use full-sample Sharpes. This is an intentional apples-to-pears comparison by design.

## Headline Results (test period 2023–2026, sorted by Sharpe)

{table_md}

## Permutation Importance Findings (RF, validation set)

| Feature | Mean Importance |
|---|---|
{imp_md}

Momentum (252-day) and volatility features dominate. Asset-class dummies carry moderate importance, confirming within-class heterogeneity is informative. Short-horizon momentum (mom_21) ranks near the bottom, matching its near-zero IC from Session 1.5B.

## Limitations

1. **Single fit:** Model trained 2003–2019, never retrained. A rolling-refit strategy (Session 2c) would better adapt to regime shifts.
2. **Short test window:** 3.3 years yields Sharpe standard errors of ~0.3–0.5; cross-strategy differences may not be statistically significant.
3. **No hyperparameter tuning:** Conservative defaults throughout. Lasso α=1e-4, RF max_depth=8, XGB lr=0.05 set without grid search.
4. **MSR instability:** Approach A amplifies estimation error in μ̂ (Michaud 1989); the long-only clip is a heuristic fix, not a solution.

## Implications

**Session 3 (DL):** MLP/LSTM/Transformer architectures should aim to beat the best Approach B Sharpe here. The weak single-fit finding suggests time-varying architectures (LSTM with rolling context, Transformer attention) may be needed.

**Session 2c (optional rolling refit):** A walk-forward refit (e.g., annual refit on expanding window) would turn the single-fit experiment into a production-grade backtesting framework and likely close the gap vs. classical baselines.
"""

with open(val_dir / 'session_2_ml_strategies.md', 'w') as f:
    f.write(val_md)
print(f'Validation doc saved: {val_dir}/session_2_ml_strategies.md')

In [ ]:
# Final summary
print('=== Session 2b Artifacts ===')
print(f'ML returns parquet:       {(out_dir / "ml_strategies_29assets.parquet").exists()}')
print(f'Comparison table CSV:     {(out_dir / "ml_strategies_comparison_table.csv").exists()}')
print(f'RF importance figure:     {(ROOT / "docs/figures/rf_permutation_importance.png").exists()}')
print(f'Validation doc:           {(val_dir / "session_2_ml_strategies.md").exists()}')
print()
print(f'Fit times — Lasso: {lasso_time:.1f}s  RF: {rf_time:.1f}s  XGB: {xgb_time:.1f}s')
print()
print('Headline table (final):')
print(fmt.to_string())

## §13 VMP Overlay on ML Strategies

VMP (volatility-managed portfolio) overlay scales each strategy's exposure inversely to its recent realized volatility, targeting the strategy's own long-run vol. This is the same overlay applied to all 31 base strategies in the paper. Wraps each of the 6 ML strategies via `apply_vmp_overlay(returns, lookback=21, clip=(0.25, 1.5))` with `target_vol = ret.std() * np.sqrt(TRADING_DAYS)`.

In [ ]:
vmp_returns = {}
for name, ret in [
    ('VMP(SignalTilt(Lasso))', ret_lasso),
    ('VMP(SignalTilt(RF))',    ret_rf),
    ('VMP(SignalTilt(XGB))',   ret_xgb),
    ('VMP(MSR(Lasso_μ̂))',     ret_msr_lasso),
    ('VMP(MSR(RF_μ̂))',        ret_msr_rf),
    ('VMP(MSR(XGB_μ̂))',       ret_msr_xgb),
]:
    target = ret.dropna().std() * np.sqrt(TRADING_DAYS)
    vmp_returns[name] = apply_vmp_overlay(ret.rename(name), lookback=21, clip=(0.25, 1.5), target_vol=target)

vmp_metrics = []
for name, r in vmp_returns.items():
    r_test = r.reindex(test_idx).dropna()
    vmp_metrics.append({
        'Strategy': name,
        'Ann Ret': ann_return(r_test),
        'Ann Vol': ann_vol(r_test),
        'Sharpe':  ann_sharpe(r_test),
        'Max DD':  max_drawdown(r_test),
    })
vmp_df = pd.DataFrame(vmp_metrics).set_index('Strategy')
print('VMP-overlay ML strategies (test period):')
print(vmp_df.round(3).to_string())

## §14 Ensemble Averaged Predictions

Simple equal-weighted average of the three model predictions: `μ̂_ens = (μ̂_lasso + μ̂_rf + μ̂_xgb) / 3`. Wired through both portfolio construction approaches — SignalTilt wrapping and MSR with ML μ̂. No retraining; pure ensemble of existing model outputs.

In [ ]:
# Build ensemble predictions
ens_predictions = (lasso_strat.predictions + rf_strat.predictions + xgb_strat.predictions) / 3.0

class _EnsembleStrategy:
    def __init__(self, predictions, tilt_strength=0.5):
        self.predictions = predictions
        self.tilt_strength = tilt_strength
    def predict_weights(self, panel, asof):
        assets = panel.universe_at(asof)
        n = len(assets)
        base_w = pd.Series(1.0/n, index=assets)
        try:
            score = self.predictions.xs(asof, level=0)
        except KeyError:
            return base_w.rename(asof)
        score = score.reindex(assets).fillna(0.0)
        std = score.std()
        zs = (score - score.mean()) / std if std > 1e-12 else pd.Series(0.0, index=assets)
        w = (base_w + self.tilt_strength * zs).clip(lower=0.0)
        total = w.sum()
        return (w / total if total > 1e-12 else base_w).rename(asof)

ens_strat = _EnsembleStrategy(ens_predictions, tilt_strength=0.5)
ret_signaltilt_ens = walk_forward_returns(ens_strat, eval_dates, returns, panel)

def _msr_walk_forward_ensemble(predictions, eval_dates, returns_wide, cov_lookback=504):
    ret_series = {}
    for d in eval_dates[:-1]:
        try:
            mu_hat = predictions.xs(d, level=0)
        except KeyError:
            continue
        hist_window = returns_wide.loc[:d].iloc[-cov_lookback:].dropna(axis=1, how='all')
        common = hist_window.columns.intersection(mu_hat.index)
        if len(common) < 5:
            continue
        cov = ledoit_wolf_cov(hist_window[common].dropna())
        cov_inv = np.linalg.pinv(cov + 1e-8 * np.eye(len(common)))
        mu = mu_hat[common].values
        w_unc = cov_inv @ mu
        w = pd.Series(w_unc, index=common).clip(lower=0)
        total = w.sum()
        if total <= 1e-12:
            w = pd.Series(1.0/len(common), index=common)
        else:
            w = w / total
        next_d_mask = returns_wide.index > d
        if not next_d_mask.any():
            continue
        next_d = returns_wide.index[next_d_mask][0]
        r_next = returns_wide.loc[next_d]
        ret_series[next_d] = (w.reindex(r_next.index).fillna(0) * r_next).sum()
    return pd.Series(ret_series)

ret_msr_ens = _msr_walk_forward_ensemble(ens_predictions, eval_dates, returns)

ens_metrics = []
for name, r in [('SignalTilt(Ensemble)', ret_signaltilt_ens), ('MSR(Ensemble_μ̂)', ret_msr_ens)]:
    r_test = r.reindex(test_idx).dropna()
    ens_metrics.append({
        'Strategy': name,
        'Ann Ret': ann_return(r_test),
        'Ann Vol': ann_vol(r_test),
        'Sharpe':  ann_sharpe(r_test),
        'Max DD':  max_drawdown(r_test),
    })
ens_df = pd.DataFrame(ens_metrics).set_index('Strategy')
print('Ensemble strategies (test period):')
print(ens_df.round(3).to_string())

## §15 Extended Comparison Table (19 strategies)

All 11 original strategies plus 6 VMP-wrapped ML variants plus 2 ensemble variants = 19 strategies sorted by Sharpe. The full picture of what ML brings to this cross-asset universe under different portfolio construction overlays.

In [ ]:
extended_rows = []
ml_base_names = {'SignalTilt(Lasso)','SignalTilt(RF)','SignalTilt(XGB)','MSR(Lasso_μ̂)','MSR(RF_μ̂)','MSR(XGB_μ̂)'}
classical_names = {'EW','MSR(LW)','SWITCH(v2a)','VMP(MDP(LW))'}
for strat_name, row in comparison.iterrows():
    if strat_name in classical_names:
        family = 'Classical'
    elif strat_name in ml_base_names:
        family = 'ML (single-fit)'
    else:
        family = 'Classical'
    extended_rows.append({
        'Strategy': strat_name, 'Ann Ret': row['Ann Ret'], 'Ann Vol': row['Ann Vol'],
        'Sharpe': row['Sharpe'], 'Max DD': row['Max DD'], 'Family': family,
    })
for name, r in vmp_returns.items():
    r_test = r.reindex(test_idx).dropna()
    extended_rows.append({
        'Strategy': name, 'Ann Ret': ann_return(r_test), 'Ann Vol': ann_vol(r_test),
        'Sharpe': ann_sharpe(r_test), 'Max DD': max_drawdown(r_test), 'Family': 'ML + VMP',
    })
for name, r in [('SignalTilt(Ensemble)', ret_signaltilt_ens), ('MSR(Ensemble_μ̂)', ret_msr_ens)]:
    r_test = r.reindex(test_idx).dropna()
    extended_rows.append({
        'Strategy': name, 'Ann Ret': ann_return(r_test), 'Ann Vol': ann_vol(r_test),
        'Sharpe': ann_sharpe(r_test), 'Max DD': max_drawdown(r_test), 'Family': 'ML (ensemble)',
    })

extended = pd.DataFrame(extended_rows).set_index('Strategy').sort_values('Sharpe', ascending=False)
print(f'Extended comparison: {len(extended)} strategies')
print(extended[['Family','Ann Ret','Ann Vol','Sharpe','Max DD']].round(3).to_string())

extended.to_csv(out_dir / 'ml_strategies_extended_comparison.csv')

ml_returns_extended = ml_returns.copy()
for name, r in vmp_returns.items():
    ml_returns_extended[name] = r.reindex(test_idx)
ml_returns_extended['SignalTilt(Ensemble)'] = ret_signaltilt_ens.reindex(test_idx)
ml_returns_extended['MSR(Ensemble_μ̂)'] = ret_msr_ens.reindex(test_idx)
ml_returns_extended.to_parquet(out_dir / 'ml_strategies_29assets_extended.parquet')
print(f'\nSaved: {out_dir}/ml_strategies_extended_comparison.csv')
print(f'Saved: {out_dir}/ml_strategies_29assets_extended.parquet ({ml_returns_extended.shape})')